<a href="https://colab.research.google.com/github/awanee02/PythonLearningJourney/blob/main/PythonDay21.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

WEEK 3 REVIEW + FIRST MAJOR CHALLENGE

First Major Challenge: Personal Finance Tracker

Project Name: FinanceFlow - Personal Finance Manager

Requirements (You must implement all):

Core Features:

Multiple Accounts (Savings, Current, Wallet)

Add Income (Salary, Freelance, Gift, etc.)

Add Expense with category
Transaction History (with date)

Dashboard showing:

Total Balance

Total Income & Total Expense this month

Top 3 expense categories

Save & Load all data automatically (using JSON)

Clean Menu-driven interface

Advanced Requirements (Must Try):

Use OOP (at least 2-3 classes)

Proper Inheritance (Base Transaction class)

Good Error Handling

Data should persist between runs

In [4]:
# Day 21 Major Challenge: FinanceFlow - Personal Finance Tracker
import json
import os
from datetime import datetime

class Transaction:
    #Represents a single income or expense
    def __init__(self, amount: float, category: str, description: str, trans_type: str):
        self.date = datetime.now().strftime("%Y-%m-%d %H:%M")
        self.amount = float(amount)
        self.category = category
        self.description = description
        self.type = trans_type  # "Income" or "Expense"

    def to_dict(self):
        return self.__dict__


class Account:
    #Represents a bank account or wallet
    def __init__(self, name: str, account_type: str):
        self.name = name
        self.account_type = account_type  # Savings, Current, Wallet
        self.balance = 0.0
        self.transactions = []

    def add_transaction(self, amount, category, description, trans_type):
        if trans_type == "Expense" and amount > self.balance:
            print(f"Insufficient balance in {self.name}!")
            return False

        transaction = Transaction(amount, category, description, trans_type)
        self.transactions.append(transaction.to_dict())

        if trans_type == "Income":
            self.balance += amount
            print(f"{amount} Income added to {self.name}")
        else:
            self.balance -= amount
            print(f"{amount} Expense recorded from {self.name}")
        return True

    def get_summary(self):
        income = sum(t['amount'] for t in self.transactions if t['type'] == "Income")
        expense = sum(t['amount'] for t in self.transactions if t['type'] == "Expense")
        return {
            "name": self.name,
            "type": self.account_type,
            "balance": self.balance,
            "total_income": income,
            "total_expense": expense,
            "transaction_count": len(self.transactions)
        }


class FinanceManager:
    """Main class to manage all accounts"""
    def __init__(self):
        self.accounts = []
        self.load_data()

    def add_account(self):
        name = input("Enter Account Name (e.g. Savings, Wallet): ")
        print("Account Types: Savings, Current, Wallet, Cash")
        acc_type = input("Enter Account Type: ").title()

        account = Account(name, acc_type)
        self.accounts.append(account)
        print(f"Account '{name}' created successfully!")

    def select_account(self):
        if not self.accounts:
            print("No accounts found! Create one first.")
            return None
        print("\nAvailable Accounts:")
        for i, acc in enumerate(self.accounts, 1):
            print(f"{i}. {acc.name} ({acc.account_type}) - ₹{acc.balance:,.2f}")
        try:
            choice = int(input("\nSelect account number: ")) - 1
            if 0 <= choice < len(self.accounts):
                return self.accounts[choice]
            else:
                print("Invalid selection!")
                return None
        except:
            print("Invalid input!")
            return None

    def add_income(self):
        account = self.select_account()
        if not account:
            return
        try:
            amount = float(input("Enter Income Amount: "))
            category = input("Enter Category (Salary, Freelance, Gift, etc.): ")
            desc = input("Enter Description: ")
            account.add_transaction(amount, category, desc, "Income")
        except ValueError:
            print("Invalid amount!")

    def add_expense(self):
        account = self.select_account()
        if not account:
            return
        try:
            amount = float(input("Enter Expense Amount: "))
            category = input("Enter Category (Food, Rent, Transport, Shopping, etc.): ")
            desc = input("Enter Description: ")
            account.add_transaction(amount, category, desc, "Expense")
        except ValueError:
            print("Invalid amount!")

    def show_dashboard(self):
        if not self.accounts:
            print("No accounts yet!")
            return

        total_balance = sum(acc.balance for acc in self.accounts)
        total_income = sum(t['amount'] for acc in self.accounts
                          for t in acc.transactions if t['type'] == "Income")
        total_expense = sum(t['amount'] for acc in self.accounts
                           for t in acc.transactions if t['type'] == "Expense")

        print("FINANCEFLOW DASHBOARD")
        print(f"Total Balance : {total_balance}")
        print(f"Total Income : {total_income}")
        print(f"Total Expense : {total_expense}")
        print(f"Net Savings : {total_income - total_expense}")

        print("\nAccount-wise Summary:")
        for acc in self.accounts:
            summary = acc.get_summary()
            print(f"• {summary['name']} ({summary['type']}) : {summary['balance']}")

    def show_transaction_history(self):
        account = self.select_account()
        if not account or not account.transactions:
            print("No transactions found!")
            return

        print(f"\nTransaction History - {account.name}")
        for t in reversed(account.transactions[-15:]):  # Last 15 transactions
            sign = "+" if t['type'] == "Income" else "-"
            print(f"{t['date']} {t['type']} {sign} {t['amount']} | {t['category']} | {t['description']}")

    def save_data(self):
        data = {
            "accounts": [
                {
                    "name": acc.name,
                    "type": acc.account_type,
                    "balance": acc.balance,
                    "transactions": acc.transactions
                } for acc in self.accounts
            ]
        }
        try:
            with open("finance_data.json", "w") as f:
                json.dump(data, f)
            print("Data saved successfully!")
        except Exception as e:
            print(f"Error saving data: {e}")

    def load_data(self):
        if os.path.exists("finance_data.json"):
            try:
                with open("finance_data.json", "r") as f:
                    data = json.load(f)
                for acc_data in data.get("accounts", []):
                    acc = Account(acc_data["name"], acc_data["type"])
                    acc.balance = acc_data["balance"]
                    acc.transactions = acc_data["transactions"]
                    self.accounts.append(acc)
                print(f"Loaded {len(self.accounts)} accounts from previous session.")
            except:
                self.accounts = []


def main():
    manager = FinanceManager()
    print("Welcome to FinanceFlow - Your Personal Finance Tracker\n")

    while True:
        print("1. Create New Account")
        print("2. Add Income")
        print("3. Add Expense")
        print("4. Show Dashboard")
        print("5. View Transaction History")
        print("6. Save Data")
        print("7. Exit")

        choice = input("\nEnter your choice: ").strip()

        if choice == "1":
            manager.add_account()
        elif choice == "2":
            manager.add_income()
        elif choice == "3":
            manager.add_expense()
        elif choice == "4":
            manager.show_dashboard()
        elif choice == "5":
            manager.show_transaction_history()
        elif choice == "6":
            manager.save_data()
        elif choice == "7":
            manager.save_data()
            print("Thank you for using FinanceFlow! Stay financially smart! 👋")
            break
        else:
            print("Invalid choice! Please select 1-7.")


if __name__ == "__main__":
    main()

Welcome to FinanceFlow - Your Personal Finance Tracker

1. Create New Account
2. Add Income
3. Add Expense
4. Show Dashboard
5. View Transaction History
6. Save Data
7. Exit

Enter your choice: 1
Enter Account Name (e.g. Savings, Wallet): Wallet
Account Types: Savings, Current, Wallet, Cash
Enter Account Type: Wallet
Account 'Wallet' created successfully!
1. Create New Account
2. Add Income
3. Add Expense
4. Show Dashboard
5. View Transaction History
6. Save Data
7. Exit

Enter your choice: 6
Data saved successfully!
1. Create New Account
2. Add Income
3. Add Expense
4. Show Dashboard
5. View Transaction History
6. Save Data
7. Exit

Enter your choice: 7
Data saved successfully!
Thank you for using FinanceFlow! Stay financially smart! 👋
